# Movie Data Analysis: Data Preparation & Cleaning

This notebook documents the data preparation and cleaning steps applied to multiple
movie-related datasets. The objective is to standardize structure, correct data types,
handle missing values, and prepare the data for analysis.

# Phase 1: Data Preparation
Prepares the raw data by loading, inspecting, and standardizing dataset structure.
It focuses on column naming, missing value assessment, and removal of unusable fields, ensuring the data is ready for further processing.

## Stage 0: Setup & Imports

In this stage, we import the required Python libraries and configure the Jupyter
environment to support data analysis and visualization.

In [35]:
# Core data analysis libraries
import numpy as np
import pandas as pd
from plotnine import *  # generally not a good thing to do to import everything from a package. However it's ok for visualization purposes in an analysis.
import os
import scipy
import warnings
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"  # to make jupyter print all outputs, not just the last one
from IPython.core.display import HTML  # to pretty print pandas df and be able to copy them over (e.g. to ppt slides)


## Stage 1: Load the Data

Each dataset represents a different aspect of movie data:
- ExpertReviews: Professional critic reviews and scores
- UserReviews: User-generated reviews and feedback
- Meta: Movie metadata such as genre, studio, and ratings
- Sales: Box office and financial performance data

In [36]:
# Load datasets
# Each dataset represents a different aspect of movie data

expert_df = pd.read_csv("../Metacritic dataset/ExpertReviews.csv")

user_df = pd.read_csv(
    "../Metacritic dataset/UserReviews.csv",
    low_memory=False  # avoids dtype inference warnings
)

meta_df = pd.read_csv("../Metacritic dataset/metaClean43Brightspace.csv")

sales_df = pd.read_csv("../Metacritic dataset/sales.csv")


## Stage 2: Initial Data Inspection

We inspect the size and structure of each dataset by checking the number of rows,
columns, and column names.

In [37]:
print("There are {} rows and {} columns in the ExpertReviews.csv ".format(expert_df.shape[0], expert_df.shape[1]))
print("There are {} rows and {} columns in the UserReviews.csv ".format(user_df.shape[0], user_df.shape[1]))
print("There are {} rows and {} columns in the metaClean43Brightspace.csv ".format(meta_df.shape[0], meta_df.shape[1]))
print("There are {} rows and {} columns in the sales.csv ".format(sales_df.shape[0], sales_df.shape[1]))

There are 238973 rows and 5 columns in the ExpertReviews.csv 
There are 319662 rows and 7 columns in the UserReviews.csv 
There are 11364 rows and 13 columns in the metaClean43Brightspace.csv 
There are 30612 rows and 16 columns in the sales.csv 


## Stage 3: Column Name Standardization

Column names are standardized to improve readability, consistency,
and ease of merging across datasets.

### 3.1 Inspect the column names

In [38]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

Column names in ExpertReviews.csv: ['url', 'idvscore', 'reviewer', 'dateP', 'Rev']
Columns names in UserReviews.csv  ['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'thumbsUp', 'thumbsTot']
Columns names in metaClean43Brightspace.csv  ['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director', 'genre', 'summary', 'awards', 'metascore', 'userscore', 'RelDate']
Columns names in sales.csv  ['year', 'release_date', 'title', 'genre', 'international_box_office', 'domestic_box_office', 'worldwide_box_office', 'production_budget', 'Unnamed: 8', 'opening_weekend', 'theatre_count', 'avg run per theatre', 'runtime', 'keywords', 'creative_type', 'url']


### 3.2 Rename the column names

In [39]:
expert_df = expert_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text'
})

user_df = user_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text',
    'thumbsUp': 'thumbs_up',
    'thumbsTot': 'thumbs_total'
})

meta_df = meta_df.rename(columns={
    'RelDate': 'release_date',
    'metascore': 'meta_score',
    'userscore': 'user_score'
})

sales_df = sales_df.rename(columns={
    'international_box_office': 'intl_box_office',
    'domestic_box_office': 'dom_box_office',
    'worldwide_box_office': 'global_box_office',
    'avg run per theatre': 'avg_run_per_theatre',
    'Unnamed: 8': 'opening_weekend_revenue'
})



### 3.3 Verify column name changes

In [40]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

Column names in ExpertReviews.csv: ['url', 'individual_score', 'reviewer', 'publish_date', 'review_text']
Columns names in UserReviews.csv  ['url', 'individual_score', 'reviewer', 'publish_date', 'review_text', 'thumbs_up', 'thumbs_total']
Columns names in metaClean43Brightspace.csv  ['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director', 'genre', 'summary', 'awards', 'meta_score', 'user_score', 'release_date']
Columns names in sales.csv  ['year', 'release_date', 'title', 'genre', 'intl_box_office', 'dom_box_office', 'global_box_office', 'production_budget', 'opening_weekend_revenue', 'opening_weekend', 'theatre_count', 'avg_run_per_theatre', 'runtime', 'keywords', 'creative_type', 'url']


## Stage 4: Missing Value Analysis

We compute the number and percentage of missing values per column
to guide data cleaning decisions.

In [41]:
def missing_summary(df, name):
    summary = (
        df.isna()
        .sum()
        .to_frame(name='Missing Values')
        .assign(Percentage=lambda x: (x['Missing Values'] / len(df) * 100).round(2))
    )
    summary = summary[summary['Missing Values'] > 0]

    print(f"\n{name} — Missing Values Summary")
    print(summary if not summary.empty else "No missing values 🎉")




In [42]:
missing_summary(expert_df, "ExpertReviews")
missing_summary(user_df, "UserReviews")
missing_summary(meta_df, "Meta")
missing_summary(sales_df, "Sales")


ExpertReviews — Missing Values Summary
                  Missing Values  Percentage
individual_score               2         0.0
reviewer                       2         0.0
publish_date                   2         0.0
review_text                    2         0.0

UserReviews — Missing Values Summary
                  Missing Values  Percentage
individual_score            3404        1.06
reviewer                    3407        1.07
publish_date                3413        1.07
review_text                 3413        1.07
thumbs_up                   3580        1.12
thumbs_total                3576        1.12

Meta — Missing Values Summary
            Missing Values  Percentage
studio                 350        3.08
rating                1067        9.39
runtime                255        2.24
cast                  3702       32.58
director                14        0.12
genre                   20        0.18
summary               5897       51.89
awards                6977       61.40


## Stage 5: Removal of Unusable Columns

Columns that contain little or no meaningful information are removed
to simplify the datasets and reduce noise.

In [43]:
# Drop fully empty columns
sales_df = sales_df.drop(columns=[
    'opening_weekend_revenue'
])

print("Columns names in sales.csv ", list(sales_df.columns))

Columns names in sales.csv  ['year', 'release_date', 'title', 'genre', 'intl_box_office', 'dom_box_office', 'global_box_office', 'production_budget', 'opening_weekend', 'theatre_count', 'avg_run_per_theatre', 'runtime', 'keywords', 'creative_type', 'url']


# Phase 2: Data Type Correction and Semantic Cleaning

Phase 2 focuses on correcting and validating data types for each dataset individually.
The goal is to ensure semantic correctness and consistency before defining relationships
or performing any dataset integration.

## Stage 0: ExpertReviews

This stage focuses on inspecting and correcting data types in the ExpertReviews dataset.

During inspection, additional issues related to missing values and text formatting
were observed. Since these issues are limited to individual columns and do not
affect table relationships, basic normalization is performed as part of Phase 2.
More advanced text processing will be addressed in a later phase.

### 0.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure and content of each column before applying any corrections.

In [44]:
# Inspect data types
print(expert_df.dtypes)

# View sample rows
expert_df.head(50)


url                  object
individual_score    float64
reviewer             object
publish_date         object
review_text          object
dtype: object


,url,individual_score,reviewer,publish_date,review_text
0,https://www.metacritic.com/movie/bronson,100.0,"""Andrew O'Hehir""",None,'Bronson owes a little or a lot to Kubrick s ...
1,https://www.metacritic.com/movie/bronson,90.0,'A.O. Scott',None,'Bronson invites you to admire its protagonis...
2,https://www.metacritic.com/movie/bronson,90.0,None,None,'Whether it s Peterson/Bronson s more theatri...
3,https://www.metacritic.com/movie/bronson,83.0,'Noel Murray',None,'There are two Bronsons on display here: the ...
4,https://www.metacritic.com/movie/bronson,80.0,'Joshua Rothkopf',None,'Refn has somehow found his way to an authent...
5,https://www.metacritic.com/movie/bronson,80.0,None,None,"'He s neither victim nor hero, but a man who,..."
6,https://www.metacritic.com/movie/bronson,80.0,'Bill Goodykoontz',None,"'This is unhinged genius, an amazing piece of..."
7,https://www.metacritic.com/movie/bronson,78.0,'Kimberley Jones',None,'Refn s artful and energetic film never goes ...
8,https://www.metacritic.com/movie/bronson,75.0,'Peter Travers',None,'This movie and Hardy s electrifying performa...
9,https://www.metacritic.com/movie/bronson,75.0,'V.A. Musetto',None,'Tom Hardy gives an amazing performance as Pe...


**Inspection summary:**

- `url` (object): Correct, used as an identifier
- `individual_score` (float64): Correct numeric type
- `reviewer` (object): Correct type, but contains missing values and inconsistent quoting
- `publish_date` (object): Incorrect type, should be datetime
- `review_text` (object): Correct type, but contains formatting artifacts and missing values

### 0.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert `publish_date` to datetime
- Normalize missing values in text fields
- Remove obvious quoting artifacts from text columns

These corrections are limited to column-level normalization and do not
alter the semantic meaning of the data.
python
Copy code


In [45]:
# Convert publish_date to datetime
expert_df['publish_date'] = pd.to_datetime(
    expert_df['publish_date'], errors='coerce'
)

# Normalize reviewer column
expert_df['reviewer'] = (
    expert_df['reviewer']
    .replace('None', pd.NA)
    .str.strip(' "\'')
)

# Normalize review_text column
expert_df['review_text'] = (
    expert_df['review_text']
    .astype(str)
    .str.strip(' "\'')
    .replace('None', pd.NA)
)


C:\Users\mhama\AppData\Local\Temp\ipykernel_29428\1668252629.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


### 0.3 Post-correction Verification

We re-inspect a sample of the data to verify that the corrections
have been applied as intended.

In [46]:
# View sample rows
expert_df.head(50)

,url,individual_score,reviewer,publish_date,review_text
0,https://www.metacritic.com/movie/bronson,100.0,Andrew O'Hehir,NaT,Bronson owes a little or a lot to Kubrick s Cl...
1,https://www.metacritic.com/movie/bronson,90.0,A.O. Scott,NaT,Bronson invites you to admire its protagonist ...
2,https://www.metacritic.com/movie/bronson,90.0,None,NaT,Whether it s Peterson/Bronson s more theatrica...
3,https://www.metacritic.com/movie/bronson,83.0,Noel Murray,NaT,There are two Bronsons on display here: the im...
4,https://www.metacritic.com/movie/bronson,80.0,Joshua Rothkopf,NaT,Refn has somehow found his way to an authentic...
5,https://www.metacritic.com/movie/bronson,80.0,None,NaT,"He s neither victim nor hero, but a man who, i..."
6,https://www.metacritic.com/movie/bronson,80.0,Bill Goodykoontz,NaT,"This is unhinged genius, an amazing piece of a..."
7,https://www.metacritic.com/movie/bronson,78.0,Kimberley Jones,NaT,Refn s artful and energetic film never goes fu...
8,https://www.metacritic.com/movie/bronson,75.0,Peter Travers,NaT,This movie and Hardy s electrifying performanc...
9,https://www.metacritic.com/movie/bronson,75.0,V.A. Musetto,NaT,Tom Hardy gives an amazing performance as Pete...


## Stage 1: UserReviews

This stage focuses on inspecting and correcting data types and basic semantic issues
in the UserReviews dataset. Compared to ExpertReviews, this table contains additional
numeric counts and more frequent missing values.

### 1.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure, content, and quality of each column before applying corrections.

In [47]:
# Inspect data types
print(user_df.dtypes)

# Inspect sample rows
user_df.head(50)


url                 object
individual_score    object
reviewer            object
publish_date        object
review_text         object
thumbs_up           object
thumbs_total        object
dtype: object


,url,individual_score,reviewer,publish_date,review_text,thumbs_up,thumbs_total
0,https://www.metacritic.com/movie/bronson,8,'Longbottom94',"'Apr 25, 2013'",'Many have dismissed this film for not explor...,2,2
1,https://www.metacritic.com/movie/bronson,9,'MartinB',"'Oct 13, 2009'",'Anyone who doesn t like this movie simply ju...,0,1
2,https://www.metacritic.com/movie/bronson,10,'Jaakko',"'Jul 19, 2012'",'Not sure what to think at this film at first...,1,1
3,https://www.metacritic.com/movie/bronson,6,'CapoR',"'Oct 13, 2009'",'Nicely portrayed but it lacks the elements t...,0,1
4,https://www.metacritic.com/movie/bronson,8,'OrwellB.',"'Oct 10, 2009'",'Bronson is more than entertainment. It is ar...,0,0
5,https://www.metacritic.com/movie/bronson,8,'RONGIU',"'Jun 10, 2011'","'Sorry, machine translation. The stage, by Ni...",0,0
6,https://www.metacritic.com/movie/bronson,8,'Tokyochuchu',"'Nov 30, 2013'","'Bronson is a great, extremely strange movie ...",0,0
7,https://www.metacritic.com/movie/bronson,7,'Fenrir81',"'Apr 22, 2011'",'Despite being flawed by a weak narrative str...,0,0
8,https://www.metacritic.com/movie/bronson,4,'cabrita',"'Aug 10, 2012'",'If valhalla rising was a weak version of agg...,0,0
9,https://www.metacritic.com/movie/bronson,10,'farknash',"'Aug 19, 2011'",'http://mikesharkey.blogspot.com/2009/04/igiz...,0,0


**Inspection summary:**

- `url` (object): Correct, used as an identifier
- `individual_score` (object): Should be numeric
- `reviewer` (object): Correct type, contains missing values
- `publish_date` (object): Should be datetime
- `review_text` (object): Correct type, contains formatting artifacts and duplication
- `thumbs_up` (object): Should be integer count
- `thumbs_total` (object): Should be integer count


### 1.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert numeric columns to appropriate numeric types
- Convert publish_date to datetime
- Normalize missing values in text columns
- Preserve text meaning while removing obvious artifacts


In [48]:
# Drop rows that contain no usable review information
user_df = user_df.dropna(
    subset=['individual_score', 'review_text', 'publish_date'],
    how='all'
)

# Convert individual_score to numeric
user_df['individual_score'] = pd.to_numeric(
    user_df['individual_score'], errors='coerce'
)

# Clean and convert publish_date
user_df['publish_date'] = (
    user_df['publish_date']
    .astype(str)
    .str.strip(' "\'')
)

user_df['publish_date'] = pd.to_datetime(
    user_df['publish_date'], errors='coerce'
)

# Convert thumbs counts to nullable integers
user_df['thumbs_up'] = pd.to_numeric(
    user_df['thumbs_up'], errors='coerce'
).astype('Int64')

user_df['thumbs_total'] = pd.to_numeric(
    user_df['thumbs_total'], errors='coerce'
).astype('Int64')

# Normalize reviewer column
user_df['reviewer'] = (
    user_df['reviewer']
    .replace('None', pd.NA)
    .str.strip(' "\'')
)

# Normalize review_text column
user_df['review_text'] = (
    user_df['review_text']
    .astype(str)
    .str.strip(' "\'')
    .replace('None', pd.NA)
)



### 1.3 Post-correction Verification

We re-inspect a sample of the data to confirm that the corrections
have been applied correctly and that no unintended changes were introduced.


In [49]:

# Verify sample rows
user_df.head(50)


,url,individual_score,reviewer,publish_date,review_text,thumbs_up,thumbs_total
0,https://www.metacritic.com/movie/bronson,8.0,Longbottom94,2013-04-25,Many have dismissed this film for not explorin...,2,2
1,https://www.metacritic.com/movie/bronson,9.0,MartinB,2009-10-13,Anyone who doesn t like this movie simply just...,0,1
2,https://www.metacritic.com/movie/bronson,10.0,Jaakko,2012-07-19,Not sure what to think at this film at first. ...,1,1
3,https://www.metacritic.com/movie/bronson,6.0,CapoR,2009-10-13,Nicely portrayed but it lacks the elements to ...,0,1
4,https://www.metacritic.com/movie/bronson,8.0,OrwellB.,2009-10-10,Bronson is more than entertainment. It is art ...,0,0
5,https://www.metacritic.com/movie/bronson,8.0,RONGIU,2011-06-10,"Sorry, machine translation. The stage, by Nico...",0,0
6,https://www.metacritic.com/movie/bronson,8.0,Tokyochuchu,2013-11-30,"Bronson is a great, extremely strange movie fr...",0,0
7,https://www.metacritic.com/movie/bronson,7.0,Fenrir81,2011-04-22,Despite being flawed by a weak narrative struc...,0,0
8,https://www.metacritic.com/movie/bronson,4.0,cabrita,2012-08-10,If valhalla rising was a weak version of aggui...,0,0
9,https://www.metacritic.com/movie/bronson,10.0,farknash,2011-08-19,http://mikesharkey.blogspot.com/2009/04/igizmo...,0,0


## Stage 2: Meta

This stage focuses on inspecting and correcting data types and basic semantic issues
in the Meta dataset. This table contains descriptive movie metadata, including
categorical variables and numeric scores, which will later be used for grouping
and analysis.


### 2.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure, content, and quality of each column before applying corrections.


In [50]:
# Inspect data types
print(meta_df.dtypes)

# Inspect sample rows
meta_df.head(50)


url              object
title            object
studio           object
rating           object
runtime         float64
cast             object
director         object
genre            object
summary          object
awards           object
meta_score        int64
user_score      float64
release_date     object
dtype: object


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,meta_score,user_score,release_date
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,01/06/2011
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,11/03/2016
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,01/12/2006
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,14/09/2012
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,28/06/2013
5,https://www.metacritic.com/movie/100-streets,100 Streets,Samuel Goldwyn Films,NaN,93.0,NaN,Jim O'Hanlon,Drama,NaN,NaN,44,6.1,13/01/2017
6,https://www.metacritic.com/movie/1000-times-go...,"1,000 Times Good Night",Film Movement,| Not Rated,117.0,NaN,Erik Poppe,Drama,NaN,NaN,57,6.8,24/10/2014
7,https://www.metacritic.com/movie/10000-bc,"10,000 BC",Warner Bros. Pictures,| PG-13,109.0,"Camilla Belle,Marco Khan,Steven Strait",Roland Emmerich,"Adventure,Drama,Fantasy",NaN,"#23MostDiscussedMovieof2008 , #27MostSharedMov...",34,4.6,07/03/2008
8,https://www.metacritic.com/movie/10000-km,"10,000 km",Broad Green Pictures,| R,99.0,NaN,Carlos Marques-Marcet,"Drama,Comedy,Romance","Two people in love, two apartments - one in Ba...",NaN,75,7.4,10/07/2015
9,https://www.metacritic.com/movie/1001-grams,1001 Grams,Kino Lorber,| Not Rated,93.0,NaN,Bent Hamer,Drama,When Norwegian scientist Marie attends a semin...,NaN,65,NaN,08/05/2015


**Inspection summary:**

Inspection of the Meta dataset shows that most text fields are semantically rich and
will be preserved for later analysis. In Phase 2, only structural corrections are
applied, including date parsing and categorical type assignment.

- `url` (object): Correct, used as an identifier
- `title` (object): Correct text field
- `studio` (object): Categorical variable, should be category
- `rating` (object): Categorical variable, should be category
- `runtime` (float64): Correct numeric type
- `cast` (object): Correct text field, contains missing values
- `director` (object): Correct text field, contains missing values
- `genre` (object): Categorical variable, should be category
- `summary` (object): Correct text field, contains missing values
- `awards` (object): Correct text field, contains many missing values
- `meta_score` (int64): Correct numeric type
- `user_score` (float64): Correct numeric type
- `release_date` (object): Should be datetime


### 2.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert release_date to datetime
- Convert categorical variables to category type
- Preserve text fields as object without semantic alteration


In [51]:
# Convert release_date to datetime
meta_df['release_date'] = pd.to_datetime(
    meta_df['release_date'], errors='coerce'
)

# Convert categorical columns to category
for col in ['studio', 'rating', 'genre']:
    meta_df[col] = meta_df[col].astype('category')


### 2.3 Post-correction Verification

We re-inspect a sample of the data to confirm that the corrections
have been applied correctly and that no unintended changes were introduced.


In [52]:
# Verify sample rows
meta_df.head(50)


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,meta_score,user_score,release_date
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-01-06
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-11-03
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-01-12
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,NaT
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,NaT
5,https://www.metacritic.com/movie/100-streets,100 Streets,Samuel Goldwyn Films,NaN,93.0,NaN,Jim O'Hanlon,Drama,NaN,NaN,44,6.1,NaT
6,https://www.metacritic.com/movie/1000-times-go...,"1,000 Times Good Night",Film Movement,| Not Rated,117.0,NaN,Erik Poppe,Drama,NaN,NaN,57,6.8,NaT
7,https://www.metacritic.com/movie/10000-bc,"10,000 BC",Warner Bros. Pictures,| PG-13,109.0,"Camilla Belle,Marco Khan,Steven Strait",Roland Emmerich,"Adventure,Drama,Fantasy",NaN,"#23MostDiscussedMovieof2008 , #27MostSharedMov...",34,4.6,2008-07-03
8,https://www.metacritic.com/movie/10000-km,"10,000 km",Broad Green Pictures,| R,99.0,NaN,Carlos Marques-Marcet,"Drama,Comedy,Romance","Two people in love, two apartments - one in Ba...",NaN,75,7.4,2015-10-07
9,https://www.metacritic.com/movie/1001-grams,1001 Grams,Kino Lorber,| Not Rated,93.0,NaN,Bent Hamer,Drama,When Norwegian scientist Marie attends a semin...,NaN,65,NaN,2015-08-05


## Stage 3: Sales

This stage focuses on inspecting and correcting data types and basic structural issues
in the Sales dataset. This table contains financial and operational information and
exhibits a high proportion of missing values, requiring careful and conservative
cleaning decisions.


In [53]:
# Inspect data types
print(sales_df.dtypes)

# Inspect sample rows
sales_df.head(50)


year                     int64
release_date            object
title                   object
genre                   object
intl_box_office        float64
dom_box_office         float64
global_box_office      float64
production_budget      float64
opening_weekend        float64
theatre_count          float64
avg_run_per_theatre    float64
runtime                float64
keywords                object
creative_type           object
url                     object
dtype: object


,year,release_date,title,genre,intl_box_office,dom_box_office,global_box_office,production_budget,opening_weekend,theatre_count,avg_run_per_theatre,runtime,keywords,creative_type,url
0,2000,January 1st,Bakha Satang,Drama,76576.0,NaN,76576.0,NaN,NaN,NaN,NaN,129.0,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Bakha-Satang...
1,2001,January 12th,Antitrust,Thriller/Suspense,6900000.0,10965209.0,17865209.0,30000000.0,5486209.0,2433.0,3.1,NaN,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Antitrust
2,2000,January 28th,Santitos,NaN,NaN,378562.0,NaN,NaN,NaN,NaN,NaN,105.0,NaN,NaN,https://www.the-numbers.com/movie/Santitos
3,2002,2002 (Wide) by,Frank McKlusky C.I.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.the-numbers.com/movie/Frank-McKlus...
4,2002,January 25th,A Walk to Remember,Drama,4833792.0,41227069.0,46060861.0,11000000.0,12177488.0,2411.0,5.3,NaN,Coming of Age,Contemporary Fiction,https://www.the-numbers.com/movie/Walk-to-Reme...
5,2002,June 21st,Zig Zag,NaN,NaN,1649.0,NaN,NaN,1649.0,1.0,1.0,NaN,NaN,NaN,https://www.the-numbers.com/movie/Zig-Zag
6,2002,May 10th,TakhtÃƒÂ© siah,Drama,NaN,21324.0,NaN,NaN,1500.0,5.0,2.0,NaN,Foreign Language,NaN,https://www.the-numbers.com/movie/Takhte-siah
7,2009,January 2nd,"Angry Monk: Reflections on Tibet, The",Documentary,NaN,2125.0,NaN,NaN,2125.0,1.0,1.0,NaN,NaN,Factual,https://www.the-numbers.com/movie/Angry-Monk-R...
8,2002,June 7th,30 Years to Life,Comedy,NaN,99023.0,NaN,NaN,3376.0,1.0,3.3,NaN,NaN,NaN,https://www.the-numbers.com/movie/30-Years-to-...
9,2008,January 2nd,The Killing of John Lennon,Drama,NaN,6975.0,NaN,NaN,3077.0,1.0,3.0,NaN,NaN,Dramatization,https://www.the-numbers.com/movie/Killing-of-J...


**Inspection summary:**

Inspection of the Sales dataset shows that the release_date column contains
inconsistent and non-standard values (e.g., partial dates and descriptive text).
To avoid introducing errors, the release_date column is preserved as text and
the year column is retained as the primary temporal reference in Phase 2.


- `year` (int64): Correct numeric type, primary temporal reference
- `release_date` (object / datetime64): Contains many missing or invalid values
- `title` (object): Correct text field
- `genre` (object): Categorical variable, should be category
- `intl_box_office` (float64): Numeric, contains many missing values
- `dom_box_office` (float64): Numeric, contains many missing values
- `global_box_office` (float64): Numeric, contains many missing values
- `production_budget` (float64): Numeric, high proportion of missing values
- `opening_weekend` (float64): Numeric, many missing values
- `theatre_count` (float64): Numeric, many missing values
- `avg_run_per_theatre` (float64): Numeric, many missing values
- `runtime` (float64): Numeric, partially missing
- `keywords` (object): Free-text field
- `creative_type` (object): Categorical variable, should be category
- `url` (object): Correct, used as an identifier



### 3.2 Data Type and Structural Corrections

Based on the inspection, we apply conservative corrections:
- Convert categorical variables to category type
- Retain numeric columns as floats to preserve missing values
- Preserve year as the primary temporal reference
- Avoid reconstructing release_date in Phase 2


In [54]:
# Convert categorical columns
sales_df['genre'] = sales_df['genre'].astype('category')
sales_df['creative_type'] = sales_df['creative_type'].astype('category')


### 3.3 Post-correction Verification

We re-inspect the data types and a sample of the data to confirm that
the corrections have been applied correctly and that no unintended changes
were introduced.


In [55]:
# Verify sample rows
sales_df.head(50)


,year,release_date,title,genre,intl_box_office,dom_box_office,global_box_office,production_budget,opening_weekend,theatre_count,avg_run_per_theatre,runtime,keywords,creative_type,url
0,2000,January 1st,Bakha Satang,Drama,76576.0,NaN,76576.0,NaN,NaN,NaN,NaN,129.0,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Bakha-Satang...
1,2001,January 12th,Antitrust,Thriller/Suspense,6900000.0,10965209.0,17865209.0,30000000.0,5486209.0,2433.0,3.1,NaN,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Antitrust
2,2000,January 28th,Santitos,NaN,NaN,378562.0,NaN,NaN,NaN,NaN,NaN,105.0,NaN,NaN,https://www.the-numbers.com/movie/Santitos
3,2002,2002 (Wide) by,Frank McKlusky C.I.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.the-numbers.com/movie/Frank-McKlus...
4,2002,January 25th,A Walk to Remember,Drama,4833792.0,41227069.0,46060861.0,11000000.0,12177488.0,2411.0,5.3,NaN,Coming of Age,Contemporary Fiction,https://www.the-numbers.com/movie/Walk-to-Reme...
5,2002,June 21st,Zig Zag,NaN,NaN,1649.0,NaN,NaN,1649.0,1.0,1.0,NaN,NaN,NaN,https://www.the-numbers.com/movie/Zig-Zag
6,2002,May 10th,TakhtÃƒÂ© siah,Drama,NaN,21324.0,NaN,NaN,1500.0,5.0,2.0,NaN,Foreign Language,NaN,https://www.the-numbers.com/movie/Takhte-siah
7,2009,January 2nd,"Angry Monk: Reflections on Tibet, The",Documentary,NaN,2125.0,NaN,NaN,2125.0,1.0,1.0,NaN,NaN,Factual,https://www.the-numbers.com/movie/Angry-Monk-R...
8,2002,June 7th,30 Years to Life,Comedy,NaN,99023.0,NaN,NaN,3376.0,1.0,3.3,NaN,NaN,NaN,https://www.the-numbers.com/movie/30-Years-to-...
9,2008,January 2nd,The Killing of John Lennon,Drama,NaN,6975.0,NaN,NaN,3077.0,1.0,3.0,NaN,NaN,Dramatization,https://www.the-numbers.com/movie/Killing-of-J...


# Phase 3: Relationships and Dataset Integration

Phase 3 focuses on identifying, validating, and defining relationships between
the datasets prepared in Phases 1 and 2. Before performing any merges, we analyze
key consistency, relationship cardinality, and potential sources of duplication
or data loss.

This phase ensures that dataset integration is performed deliberately and
correctly, rather than implicitly or through trial and error.


## Stage 0: Meta ↔ Sales Relationship Feasibility

The Sales dataset originates from a different source and does not share a
common URL structure with Meta. To evaluate whether a relationship can exist,
we perform a title-based intersection analysis without merging any data.

Meta is treated as the anchor table.


### 0.1 Title-based Intersection (NO MERGE)

Movie titles are normalized and compared to identify Meta movies that
also appear in the Sales dataset.


In [56]:
# Normalize titles
meta_titles = (
    meta_df['title']
    .astype(str)
    .str.lower()
    .str.strip()
)

sales_titles = (
    sales_df['title']
    .astype(str)
    .str.lower()
    .str.strip()
)

# Create Sales title set
sales_title_set = set(sales_titles.dropna())

# Identify Meta titles that exist in Sales
meta_in_sales_mask = meta_titles.isin(sales_title_set)

# Summary
print("Meta movies with a Sales match:", meta_in_sales_mask.sum())
print(
    "Percentage of Meta movies covered by Sales:",
    f"{meta_in_sales_mask.mean() * 100:.2f}%"
)

# Inspect intersecting Meta rows
meta_df.loc[meta_in_sales_mask, ['title', 'release_date']].head(20)


Meta movies with a Sales match: 8643
Percentage of Meta movies covered by Sales: 76.06%


,title,release_date
1,10 Cloverfield Lane,2016-11-03
3,10 Years,NaT
4,100 Bloody Acres,NaT
5,100 Streets,NaT
8,"10,000 km",2015-10-07
9,1001 Grams,2015-08-05
11,102 Dalmatians,NaT
12,10th & Wolf,NaT
16,12,2009-04-03
18,12 Hour Shift,2020-02-10


## Stage 1: Meta ↔ ExpertReviews Relationship

The Meta and ExpertReviews datasets share a common URL identifier.
This stage validates coverage and confirms the expected one-to-many
relationship between movies and expert reviews.


### 1.1 URL-based Relationship Inspection (NO MERGE)


In [57]:
meta_urls = set(meta_df['url'])
expert_urls = set(expert_df['url'])

# Intersection
common_urls = meta_urls.intersection(expert_urls)

print("Meta movies:", len(meta_urls))
print("Expert review movies:", len(expert_urls))
print("Common URLs:", len(common_urls))

print(
    "Percentage of Meta movies with ExpertReviews:",
    f"{len(common_urls) / len(meta_urls) * 100:.2f}%"
)

# Cardinality: number of expert reviews per movie
expert_review_counts = expert_df['url'].value_counts()

print("\nExpert review count summary:")
print(expert_review_counts.describe())


Meta movies: 11364
Expert review movies: 11364
Common URLs: 11364
Percentage of Meta movies with ExpertReviews: 100.00%

Expert review count summary:
count    11364.000000
mean        21.028951
std         11.590415
min          1.000000
25%         11.000000
50%         18.000000
75%         30.000000
max         67.000000
Name: count, dtype: float64


## Stage 2: Meta ↔ UserReviews Relationship

The Meta and UserReviews datasets also share a common URL identifier.
However, user reviews are optional, meaning not all movies receive
user-generated feedback. This stage validates coverage and review volume.


### 2.1 URL-based Relationship Inspection (NO MERGE)


In [58]:
meta_urls = set(meta_df['url'])
user_urls = set(user_df['url'])

# Intersection
common_urls = meta_urls.intersection(user_urls)

print("Meta movies:", len(meta_urls))
print("User review movies:", len(user_urls))
print("Common URLs:", len(common_urls))

print(
    "Percentage of Meta movies with UserReviews:",
    f"{len(common_urls) / len(meta_urls) * 100:.2f}%"
)

# Cardinality: number of user reviews per movie
user_review_counts = user_df['url'].value_counts()

print("\nUser review count summary:")
print(user_review_counts.describe())


Meta movies: 11364
User review movies: 8896
Common URLs: 8896
Percentage of Meta movies with UserReviews: 78.28%

User review count summary:
count    8896.000000
mean       35.550585
std       101.886194
min         1.000000
25%         2.000000
50%         8.000000
75%        29.000000
max      2763.000000
Name: count, dtype: float64


## Phase 3 Summary

- Meta serves as the anchor table, representing one row per movie.
- ExpertReviews and UserReviews exhibit clear one-to-many relationships
  with Meta using the URL identifier.
- The Sales dataset cannot be directly joined via URL but shows meaningful
  overlap when titles are used as a semantic linking key.
- No merges are performed in this phase; all relationships are evaluated
  conceptually and quantitatively to inform later integration decisions.
